In [ ]:
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy import stats
from patsy import build_design_matrices


In [ ]:
# ==============================
# Editable configuration
# ==============================

# "single" requires exactly one match; "folder" processes every match.
INPUT_MODE = "single"

# "blank_separated_columns", "long_table", or "graphpad_two_header"
INPUT_TABLE_FORMAT = "blank_separated_columns"

# Directory containing the input file(s). Replace this anonymous placeholder.
INPUT_FOLDER = r"path_to_your_data"

# Examples: "*.csv", "*VIP*.csv", or "*graphpad_nested_two_header.xlsx"
GLOB_PATTERN = "*.csv"

# LMM + rank-transformed LMM with GraphPad nested tables

This notebook supports three input formats:

1. **blank_separated_columns**: each CSV column is one group/condition, and blank cells inside each column separate mice.
2. **long_table**: an existing long table with one row per neuron/event, using the helper defaults group, mouse_id, and value.
3. **graphpad_two_header**: a GraphPad-style nested matrix such as graphpad_nested_two_header.xlsx, where row 1 is group/condition and row 2 is mouse ID.

The editable configuration is intentionally limited to input mode, table format, folder, and file-matching pattern. In single mode, the folder and pattern must match exactly one file; in folder mode, every match is processed.

It then runs only two mixed models:

1. **Raw-value LMM**: value ~ group + (1 | mouse)
2. **Rank-transformed LMM**: rank_value ~ group + (1 | mouse)

Outputs are intentionally focused:

- long_table.csv
- ranked_long_table.csv
- test_results.xlsx
- graphpad_nested_two_header.xlsx

The GraphPad workbook uses a two-row header: row 1 is group/condition and row 2 is mouse ID. The rank transformation is recomputed globally across all included rows before fitting the rank-transformed LMM.

In [ ]:
# ==============================
# Helpers: IO and parsing
# ==============================

def read_csv_with_encoding_fallback(csv_path, encoding_order=("utf-8-sig", "utf-8", "gbk")):
    csv_path = Path(csv_path)
    last_error = None

    for enc in encoding_order:
        try:
            df = pd.read_csv(csv_path, encoding=enc)
            print(f"Loaded CSV with encoding: {enc}")
            return df
        except UnicodeDecodeError as e:
            last_error = e
            print(f"Encoding fallback triggered: {enc} failed, trying next encoding.")
        except Exception as e:
            last_error = e
            print(f"CSV read failed with encoding {enc}: {e}")
            print("Trying next encoding if available.")

    raise RuntimeError(f"Failed to read CSV after trying encodings: {encoding_order}") from last_error



def read_table_file(input_path, encoding_order=("utf-8-sig", "utf-8", "gbk"), sheet_name=None, header="infer"):
    """
    Read CSV/TSV/XLSX files.

    For graphpad_two_header xlsx input, pass header=None and sheet_name="raw_for_GraphPad".
    """
    input_path = Path(input_path)
    suffix = input_path.suffix.lower()

    if suffix in [".xlsx", ".xls"]:
        if sheet_name is None:
            sheet_name = 0

        # pandas.read_excel does not accept header="infer".
        # For ordinary Excel tables, use the first row as column names.
        # For GraphPad two-header matrices, callers pass header=None so string mouse IDs
        # in the second header row are preserved as data and parsed explicitly later.
        excel_header = 0 if header == "infer" else header

        df = pd.read_excel(input_path, sheet_name=sheet_name, header=excel_header, engine="openpyxl")
        print(f"Loaded Excel file: {input_path.name}, sheet={sheet_name}, header={excel_header}")
        return df

    if suffix == ".tsv":
        last_error = None
        for enc in encoding_order:
            try:
                df = pd.read_csv(input_path, sep="\t", encoding=enc, header=header)
                print(f"Loaded TSV with encoding: {enc}")
                return df
            except UnicodeDecodeError as e:
                last_error = e
                print(f"Encoding fallback triggered: {enc} failed, trying next encoding.")
            except Exception as e:
                last_error = e
                print(f"TSV read failed with encoding {enc}: {e}")
                print("Trying next encoding if available.")
        raise RuntimeError(f"Failed to read TSV after trying encodings: {encoding_order}") from last_error

    if header == "infer":
        return read_csv_with_encoding_fallback(input_path, encoding_order)

    last_error = None
    for enc in encoding_order:
        try:
            df = pd.read_csv(input_path, encoding=enc, header=header)
            print(f"Loaded CSV with encoding: {enc}")
            return df
        except UnicodeDecodeError as e:
            last_error = e
            print(f"Encoding fallback triggered: {enc} failed, trying next encoding.")
        except Exception as e:
            last_error = e
            print(f"CSV read failed with encoding {enc}: {e}")
            print("Trying next encoding if available.")
    raise RuntimeError(f"Failed to read CSV after trying encodings: {encoding_order}") from last_error


def parse_graphpad_two_header_matrix(
    matrix_df,
    group_order=None,
    group_header_row=0,
    mouse_header_row=1,
    data_start_row=2,
):
    """
    Parse GraphPad-style nested matrix into long_table.

    Expected structure:
    row group_header_row : group / condition names
    row mouse_header_row : mouse IDs
    rows data_start_row:  neuron/event values

    Each column is one group-mouse nested subcolumn.
    """
    rows = []
    excluded_cells = []
    col_specs = []

    n_rows, n_cols = matrix_df.shape

    if n_rows <= max(group_header_row, mouse_header_row, data_start_row - 1):
        raise ValueError("GraphPad two-header input has too few rows.")

    for col_i in range(n_cols):
        group = matrix_df.iat[group_header_row, col_i]
        mouse_id = matrix_df.iat[mouse_header_row, col_i]

        if is_blank_cell(group) and is_blank_cell(mouse_id):
            continue

        if is_blank_cell(group) or is_blank_cell(mouse_id):
            print(
                f"GraphPad input fallback: column {col_i + 1} skipped because "
                f"group or mouse_id header is blank. group={group}, mouse_id={mouse_id}"
            )
            continue

        group = str(group).strip()
        mouse_id = str(mouse_id).strip()
        nested_subject = f"{group}_{mouse_id}"
        col_specs.append((col_i, group, mouse_id, nested_subject))

    if len(col_specs) == 0:
        raise ValueError("No valid group-mouse columns found in GraphPad two-header input.")

    if group_order is None:
        group_names = []
        for _, group, _, _ in col_specs:
            if group not in group_names:
                group_names.append(group)
    else:
        group_names = list(group_order)
        extra_groups = []
        for _, group, _, _ in col_specs:
            if group not in group_names and group not in extra_groups:
                extra_groups.append(group)
        if extra_groups:
            print(
                "GraphPad input fallback: groups not listed in GROUP_ORDER were retained "
                f"after GROUP_ORDER: {extra_groups}"
            )
            group_names = group_names + extra_groups

    for col_i, group, mouse_id, nested_subject in col_specs:
        for row_i in range(data_start_row, n_rows):
            cell = matrix_df.iat[row_i, col_i]

            if is_blank_cell(cell):
                continue

            numeric_value = try_float(cell)
            if numeric_value is None or not np.isfinite(numeric_value):
                excluded_cells.append({
                    "excel_row_1based": row_i + 1,
                    "excel_col_1based": col_i + 1,
                    "group": group,
                    "mouse_id": mouse_id,
                    "raw_value": cell,
                    "reason": "non-numeric or non-finite cell excluded",
                })
                print(
                    f"Excluded non-numeric GraphPad cell: row={row_i + 1}, "
                    f"col={col_i + 1}, group={group}, mouse_id={mouse_id}, value={cell}"
                )
                continue

            rows.append({
                "group": group,
                "mouse_id": mouse_id,
                "nested_subject": nested_subject,
                "source_row": row_i + 1,
                "source_col": col_i + 1,
                "value": numeric_value,
            })

    long_table = pd.DataFrame(rows)
    excluded_cells_df = pd.DataFrame(excluded_cells)

    if long_table.empty:
        raise ValueError("No valid numeric values were parsed from GraphPad two-header input.")

    if not excluded_cells_df.empty:
        print("GraphPad input cell exclusion occurred:")
        print(excluded_cells_df.head(20))

    long_table["group"] = pd.Categorical(long_table["group"].astype(str), categories=group_names, ordered=True)
    long_table["mouse_id"] = long_table["mouse_id"].astype(str)
    long_table["nested_subject"] = long_table["nested_subject"].astype(str)

    print(
        f"GraphPad two-header input parsed: {len(long_table)} values, "
        f"{len(group_names)} groups, {long_table['nested_subject'].nunique()} group-mouse columns."
    )

    return long_table, group_names



def is_blank_cell(x):
    if pd.isna(x):
        return True
    if isinstance(x, str) and x.strip() == "":
        return True
    return False


def try_float(x):
    try:
        return float(x)
    except Exception:
        return None


def parse_blank_separated_columns(raw_df, group_order=None, mouse_prefix="M", min_values_per_mouse=1):
    if group_order is None:
        group_names = list(raw_df.columns)
    else:
        missing = [g for g in group_order if g not in raw_df.columns]
        if missing:
            raise ValueError(f"GROUP_ORDER contains columns not found in CSV: {missing}")
        group_names = list(group_order)

    rows = []
    excluded_segments = []
    excluded_cells = []

    for group in group_names:
        col_values = raw_df[group].tolist()
        current_values = []
        current_source_rows = []
        mouse_counter = 0

        def close_current_segment():
            nonlocal mouse_counter, current_values, current_source_rows
            if len(current_values) == 0:
                return

            if len(current_values) < min_values_per_mouse:
                excluded_segments.append({
                    "group": group,
                    "mouse_index_candidate": mouse_counter + 1,
                    "n_values": len(current_values),
                    "reason": f"segment shorter than MIN_VALUES_PER_MOUSE={min_values_per_mouse}",
                    "source_rows": ";".join(map(str, current_source_rows)),
                })
            else:
                mouse_counter += 1
                mouse_id = f"{mouse_prefix}{mouse_counter}"
                nested_subject = f"{group}_{mouse_id}"
                for value, source_row in zip(current_values, current_source_rows):
                    rows.append({
                        "group": group,
                        "mouse_id": mouse_id,
                        "nested_subject": nested_subject,
                        "source_row": source_row,
                        "value": value,
                    })

            current_values = []
            current_source_rows = []

        for row_index_0, cell in enumerate(col_values):
            # source_row is 1-based row index in the CSV body plus header row.
            # In Excel/CSV visual terms, the first data row is often row 2.
            source_row = row_index_0 + 2

            if is_blank_cell(cell):
                close_current_segment()
                continue

            numeric_value = try_float(cell)
            if numeric_value is None or not np.isfinite(numeric_value):
                excluded_cells.append({
                    "group": group,
                    "source_row": source_row,
                    "raw_value": cell,
                    "reason": "non-numeric or non-finite cell excluded",
                })
                print(f"Excluded non-numeric cell: group={group}, source_row={source_row}, value={cell}")
                continue

            current_values.append(numeric_value)
            current_source_rows.append(source_row)

        close_current_segment()

    long_table = pd.DataFrame(rows)
    excluded_segments_df = pd.DataFrame(excluded_segments)
    excluded_cells_df = pd.DataFrame(excluded_cells)

    if long_table.empty:
        raise ValueError("No valid numeric data were parsed. Check the CSV structure and blank separators.")

    if not excluded_segments_df.empty:
        print("Segment exclusion occurred:")
        print(excluded_segments_df)

    if not excluded_cells_df.empty:
        print("Cell exclusion occurred:")
        print(excluded_cells_df)

    long_table["group"] = pd.Categorical(long_table["group"], categories=group_names, ordered=True)
    long_table["mouse_id"] = long_table["mouse_id"].astype(str)
    long_table["nested_subject"] = long_table["nested_subject"].astype(str)

    return long_table, group_names


def standardize_existing_long_table(
    raw_df,
    group_col="group",
    mouse_col="mouse_id",
    value_col="value",
    nested_subject_col=None,
    group_order=None,
):
    """
    Standardize an existing long table to the columns required by this notebook:
    group, mouse_id, nested_subject, source_row, value.

    Required input columns: group_col, mouse_col, value_col.
    Optional input column: nested_subject_col.
    """
    required_cols = [group_col, mouse_col, value_col]
    missing = [c for c in required_cols if c not in raw_df.columns]
    if missing:
        raise ValueError(f"Long-table input is missing required columns: {missing}")

    long_table = pd.DataFrame()
    long_table["group"] = raw_df[group_col].astype(str)
    long_table["mouse_id"] = raw_df[mouse_col].astype(str)
    long_table["value"] = pd.to_numeric(raw_df[value_col], errors="coerce")

    if "source_row" in raw_df.columns:
        long_table["source_row"] = raw_df["source_row"]
    else:
        # Excel-like row number: header is row 1, first data row is row 2.
        long_table["source_row"] = np.arange(len(raw_df)) + 2

    if nested_subject_col is not None and nested_subject_col in raw_df.columns:
        long_table["nested_subject"] = raw_df[nested_subject_col].astype(str)
    else:
        print("Long-table fallback: nested_subject was generated as group + '_' + mouse_id.")
        long_table["nested_subject"] = long_table["group"].astype(str) + "_" + long_table["mouse_id"].astype(str)

    before_n = len(long_table)
    invalid_mask = (
        long_table["group"].isna()
        | long_table["mouse_id"].isna()
        | long_table["value"].isna()
        | ~np.isfinite(long_table["value"])
    )

    if invalid_mask.any():
        excluded = long_table.loc[invalid_mask].copy()
        print(f"Long-table exclusion: {len(excluded)} rows excluded because group/mouse/value was invalid.")
        print(excluded.head(20))

    long_table = long_table.loc[~invalid_mask].copy()

    if long_table.empty:
        raise ValueError("No valid rows remain after standardizing the long table.")

    if group_order is None:
        group_names = list(pd.unique(long_table["group"].astype(str)))
    else:
        group_names = list(group_order)
        missing_groups = sorted(set(long_table["group"].astype(str)) - set(group_names))
        if missing_groups:
            print(
                "Long-table fallback: rows from groups not listed in GROUP_ORDER were retained, "
                f"but they will appear after GROUP_ORDER: {missing_groups}"
            )
            group_names = group_names + missing_groups

    long_table["group"] = pd.Categorical(long_table["group"].astype(str), categories=group_names, ordered=True)
    long_table["mouse_id"] = long_table["mouse_id"].astype(str)
    long_table["nested_subject"] = long_table["nested_subject"].astype(str)

    print(
        f"Long-table input standardized: {len(long_table)} valid rows retained "
        f"from {before_n} original rows."
    )

    return long_table, group_names


In [ ]:
# ==============================
# Helpers: GraphPad two-header nested table
# ==============================

def natural_mouse_sort_key(mouse_id):
    text = str(mouse_id)
    prefix = "".join([c for c in text if not c.isdigit()])
    digits = "".join([c for c in text if c.isdigit()])
    if digits:
        return (prefix, int(digits), text)
    return (prefix, float("inf"), text)


def build_graphpad_two_header_table(long_df, value_col, group_names):
    """
    Build a GraphPad-friendly nested matrix with two header rows.

    Row 1: group / condition names
    Row 2: mouse IDs
    Body : neuron/event values for each group-mouse column, padded with blanks

    This table is for manual copy-paste/import into GraphPad nested tables.
    """
    required = {"group", "mouse_id", value_col}
    missing = sorted(required - set(long_df.columns))
    if missing:
        raise ValueError(f"Missing required columns for GraphPad table: {missing}")

    col_specs = []

    for group in group_names:
        group_mask = long_df["group"].astype(str) == str(group)
        group_df = long_df.loc[group_mask].copy()

        if group_df.empty:
            print(f"GraphPad table fallback: group {group} has no rows and is skipped.")
            continue

        # Preserve mouse order when possible, then sort naturally.
        mice = list(pd.unique(group_df["mouse_id"].astype(str)))
        mice = sorted(mice, key=natural_mouse_sort_key)

        for mouse_id in mice:
            values = (
                group_df.loc[group_df["mouse_id"].astype(str) == str(mouse_id), value_col]
                .dropna()
                .tolist()
            )
            if len(values) == 0:
                print(f"GraphPad table fallback: {group}-{mouse_id} has no valid values and is skipped.")
                continue

            col_specs.append({
                "group": str(group),
                "mouse_id": str(mouse_id),
                "values": values,
                "n_values": len(values),
            })

    if len(col_specs) == 0:
        raise ValueError("No valid group-mouse columns were available for the GraphPad table.")

    max_len = max(spec["n_values"] for spec in col_specs)

    header_group = [spec["group"] for spec in col_specs]
    header_mouse = [spec["mouse_id"] for spec in col_specs]

    body_rows = []
    for row_i in range(max_len):
        row = []
        for spec in col_specs:
            if row_i < spec["n_values"]:
                row.append(spec["values"][row_i])
            else:
                row.append("")
        body_rows.append(row)

    copy_table = pd.DataFrame([header_group, header_mouse] + body_rows)

    summary = pd.DataFrame([
        {
            "group": spec["group"],
            "mouse_id": spec["mouse_id"],
            "n_values": spec["n_values"],
        }
        for spec in col_specs
    ])

    return copy_table, summary


def build_random_effect_summary_wide_table(long_df, value_col, group_names, random_effect_col):
    """
    Build a wide table summarizing each random-effect unit within each fixed-effect level.

    For each group and random-effect unit, Mean / SEM / N are calculated from the
    observations that entered the LMM for the selected outcome. N is the number of
    observations inside that random-effect unit, not the number of random-effect units.
    """
    required = {"group", random_effect_col, value_col}
    missing = sorted(required - set(long_df.columns))
    if missing:
        raise ValueError(f"Missing required columns for random-effect summary: {missing}")

    group_blocks = []
    max_rows = 0

    for group in group_names:
        group_mask = long_df["group"].astype(str) == str(group)
        group_df = long_df.loc[group_mask].copy()

        if group_df.empty:
            print(f"Random-effect summary fallback: group {group} has no rows and is skipped.")
            continue

        summary = (
            group_df
            .groupby(random_effect_col, observed=True)[value_col]
            .agg(Mean="mean", SD="std", N="count")
            .reset_index()
            .rename(columns={random_effect_col: "ID"})
        )
        summary["SEM"] = summary["SD"] / np.sqrt(summary["N"])
        summary = summary[["ID", "Mean", "SEM", "N"]]
        summary["ID"] = summary["ID"].astype(str)
        summary = summary.sort_values("ID", key=lambda s: s.map(natural_mouse_sort_key)).reset_index(drop=True)

        group_blocks.append((str(group), summary))
        max_rows = max(max_rows, summary.shape[0])

    if not group_blocks:
        raise ValueError("No valid groups were available for the random-effect summary table.")

    header_group = []
    header_stat = []
    body_rows = []

    for group, _ in group_blocks:
        header_group.extend([group, group, group, group])
        header_stat.extend(["ID", "Mean", "SEM", "N"])

    for row_i in range(max_rows):
        row = []
        for _, summary in group_blocks:
            if row_i < summary.shape[0]:
                row.extend(summary.loc[row_i, ["ID", "Mean", "SEM", "N"]].tolist())
            else:
                row.extend(["", "", "", ""])
        body_rows.append(row)

    return pd.DataFrame([header_group, header_stat] + body_rows)


def save_graphpad_two_header_outputs(raw_table, raw_summary, rank_table, rank_summary, output_dir,prefix):
    output_dir = Path(output_dir)

    if 'graphpad' in prefix:
        xlsx_path = output_dir / f"{prefix}.xlsx"
    else:
        xlsx_path = output_dir / f"{prefix}_graphpad_nested_two_header.xlsx"



    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        raw_table.to_excel(writer, sheet_name="raw_for_GraphPad", header=False, index=False)
        rank_table.to_excel(writer, sheet_name="rank_for_GraphPad", header=False, index=False)
        raw_summary.to_excel(writer, sheet_name="raw_column_summary", index=False)
        rank_summary.to_excel(writer, sheet_name="rank_column_summary", index=False)

    return {
        "xlsx": xlsx_path,
    }

In [ ]:
# ==============================
# Helpers: LMM fitting and pairwise comparisons
# ==============================

def fit_mixedlm_with_fallback(df, formula, group_vector, maxiter=2000, optimizer_order=None):
    if optimizer_order is None:
        optimizer_order = ["lbfgs", "powell", "cg", "bfgs"]

    last_error = None

    for method in optimizer_order:
        try:
            with warnings.catch_warnings(record=True) as caught_warnings:
                warnings.simplefilter("always")
                model = smf.mixedlm(formula, data=df, groups=group_vector)
                result = model.fit(reml=False, method=method, maxiter=maxiter, disp=False)

                if caught_warnings:
                    print(f"MixedLM warning with optimizer={method}:")
                    for w in caught_warnings[-3:]:
                        print(f"  {w.message}")

            print(f"MixedLM fitted successfully with optimizer={method}")
            return result, method, "success"

        except Exception as e:
            last_error = e
            print(f"MixedLM fallback triggered: optimizer={method} failed with error: {e}")

    raise RuntimeError("All MixedLM optimizers failed.") from last_error


def likelihood_ratio_test(full_result, null_result):
    lr_stat = 2 * (full_result.llf - null_result.llf)
    df_diff = int(full_result.df_modelwc - null_result.df_modelwc)

    if df_diff <= 0:
        print("Fallback triggered: df_diff <= 0 for LRT. Setting P value to NaN.")
        p_value = np.nan
    else:
        p_value = stats.chi2.sf(lr_stat, df_diff)

    return lr_stat, df_diff, p_value


def extract_random_variance(result):
    try:
        random_intercept_variance = float(result.cov_re.iloc[0, 0])
    except Exception:
        print("Fallback triggered: random-effect variance could not be extracted. Setting to NaN.")
        random_intercept_variance = np.nan

    try:
        residual_variance = float(result.scale)
    except Exception:
        print("Fallback triggered: residual variance could not be extracted. Setting to NaN.")
        residual_variance = np.nan

    return random_intercept_variance, residual_variance


def fixed_effects_table(result):
    out = pd.DataFrame({
        "term": result.fe_params.index,
        "estimate": result.fe_params.values,
        "se": result.bse_fe.values,
        "z": result.fe_params.values / result.bse_fe.values,
        "p": result.pvalues[result.fe_params.index].values,
    })
    return out


def pairwise_group_comparisons(result, df, outcome_col, random_effect_col, group_names, p_adjust_method="holm"):
    design_info = result.model.data.design_info
    grid = pd.DataFrame({
        "group": pd.Categorical(group_names, categories=group_names, ordered=True)
    })

    model_df = result.model.data.frame.copy()
    model_df["_lmm_random_effect_unit"] = result.model.groups

    unit_level_values = (
        model_df
        .groupby(["group", "_lmm_random_effect_unit"], observed=True)[outcome_col]
        .mean()
        .reset_index(name="unit_mean")
    )
    group_stats = (
        unit_level_values
        .groupby("group", observed=True)["unit_mean"]
        .agg(group_N="count", group_mean="mean", group_SD="std")
    )
    group_stats["group_SEM"] = group_stats["group_SD"] / np.sqrt(group_stats["group_N"])
    group_stats = group_stats.to_dict(orient="index")

    x_grid = np.asarray(build_design_matrices([design_info], grid)[0])
    beta = result.fe_params.values

    cov_all = result.cov_params()
    fe_names = list(result.fe_params.index)
    cov_beta = cov_all.loc[fe_names, fe_names].to_numpy()

    rows = []
    for i, j in itertools.combinations(range(len(group_names)), 2):
        group_1 = group_names[i]
        group_2 = group_names[j]
        contrast = x_grid[i, :] - x_grid[j, :]
        estimate = float(contrast @ beta)
        se = float(np.sqrt(contrast @ cov_beta @ contrast.T))

        if se <= 0 or not np.isfinite(se):
            print(f"Fallback triggered: invalid SE for comparison {group_1} vs {group_2}.")
            z_value = np.nan
            p_raw = np.nan
        else:
            z_value = estimate / se
            p_raw = 2 * stats.norm.sf(abs(z_value))

        group_1_stats = group_stats.get(group_1, {})
        group_2_stats = group_stats.get(group_2, {})

        rows.append({
            "comparison": f"{group_1} vs {group_2}",
            "group_1": group_1,
            "group_2": group_2,
            "group_1_mean": group_1_stats.get("group_mean", np.nan),
            "group_2_mean": group_2_stats.get("group_mean", np.nan),
            "group_1_SD": group_1_stats.get("group_SD", np.nan),
            "group_2_SD": group_2_stats.get("group_SD", np.nan),
            "group_1_SEM": group_1_stats.get("group_SEM", np.nan),
            "group_2_SEM": group_2_stats.get("group_SEM", np.nan),
            "group_1_N": group_1_stats.get("group_N", 0),
            "group_2_N": group_2_stats.get("group_N", 0),
            "summary_definition": f"mean/SD/SEM calculated from per-{random_effect_col} means used by the fitted LMM",
            "estimate_group1_minus_group2": estimate,
            "se": se,
            "z": z_value,
            "p_raw": p_raw,
        })

    pairwise_df = pd.DataFrame(rows)

    valid_mask = pairwise_df["p_raw"].notna()
    pairwise_df["p_adjusted"] = np.nan

    if valid_mask.any():
        _, p_adj, _, _ = multipletests(
            pairwise_df.loc[valid_mask, "p_raw"].values,
            method=p_adjust_method
        )
        pairwise_df.loc[valid_mask, "p_adjusted"] = p_adj
    else:
        print("Fallback triggered: no valid pairwise P values to adjust.")

    pairwise_df["p_adjust_method"] = p_adjust_method
    return pairwise_df


def run_lmm_test(df, outcome_col, group_names, label, random_effect_col="mouse_id", p_adjust_method="holm", maxiter=2000, optimizer_order=None):
    required = {"group", random_effect_col, outcome_col}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Missing required columns for {label}: {missing}")

    formula_full = f"{outcome_col} ~ C(group)"
    formula_null = f"{outcome_col} ~ 1"

    group_vector = df[random_effect_col]

    full_result, full_optimizer, full_status = fit_mixedlm_with_fallback(
        df=df,
        formula=formula_full,
        group_vector=group_vector,
        maxiter=maxiter,
        optimizer_order=optimizer_order,
    )

    null_result, null_optimizer, null_status = fit_mixedlm_with_fallback(
        df=df,
        formula=formula_null,
        group_vector=group_vector,
        maxiter=maxiter,
        optimizer_order=optimizer_order,
    )

    lr_stat, df_diff, p_value = likelihood_ratio_test(full_result, null_result)
    random_var, residual_var = extract_random_variance(full_result)

    omnibus = pd.DataFrame([{
        "model": label,
        "outcome": outcome_col,
        "formula_full": formula_full,
        "formula_null": formula_null,
        "random_effect_col": random_effect_col,
        "n_observations": int(df.shape[0]),
        "n_groups": int(len(group_names)),
        "n_random_effect_units": int(df[random_effect_col].nunique()),
        "full_optimizer": full_optimizer,
        "null_optimizer": null_optimizer,
        "lrt_chi2": lr_stat,
        "df": df_diff,
        "p": p_value,
        "random_intercept_variance": random_var,
        "residual_variance": residual_var,
        "loglik_full": full_result.llf,
        "loglik_null": null_result.llf,
    }])

    fixed = fixed_effects_table(full_result)
    fixed.insert(0, "model", label)

    pairwise = pairwise_group_comparisons(
        result=full_result,
        df=df,
        outcome_col=outcome_col,
        random_effect_col=random_effect_col,
        group_names=group_names,
        p_adjust_method=p_adjust_method,
    )
    pairwise.insert(0, "model", label)

    return {
        "omnibus": omnibus,
        "fixed_effects": fixed,
        "pairwise": pairwise,
        "full_result": full_result,
        "null_result": null_result,
    },full_result,null_result
def statsmodels_summary_to_dataframe(result, label):
    """
    Convert statsmodels result.summary().as_text() into a one-column DataFrame.
    This preserves the full human-readable summary in Excel.
    """
    try:
        summary_text = result.summary().as_text()
    except Exception as e:
        summary_text = f"Failed to export summary for {label}: {e}"

    lines = summary_text.splitlines()
    return pd.DataFrame({
        "model": [label] * len(lines),
        "summary_text": lines,
    })

In [ ]:
# ==============================
# Main run
# ==============================

input_folder = Path(INPUT_FOLDER)
output_root = input_folder / "lmm_rank_lmm_outputs"
output_root.mkdir(parents=True, exist_ok=True)

matched_files = sorted(input_folder.rglob(GLOB_PATTERN))

if INPUT_MODE == "single":
    if len(matched_files) != 1:
        raise ValueError(
            "INPUT_MODE='single' requires exactly one file matched by "
            f"INPUT_FOLDER + GLOB_PATTERN; found {len(matched_files)}."
        )
    input_files = matched_files
elif INPUT_MODE == "folder":
    input_files = matched_files
else:
    raise ValueError('INPUT_MODE must be either "single" or "folder".')

if len(input_files) == 0:
    raise FileNotFoundError("No input files found. Check INPUT_FOLDER and GLOB_PATTERN.")

all_dataset_summaries = []

for input_file in input_files:
    print("\n" + "=" * 80)
    print(f"Processing: {input_file}")

    dataset_name = Path(input_file).stem
    dataset_output_dir = output_root / dataset_name
    dataset_output_dir.mkdir(parents=True, exist_ok=True)

    if INPUT_TABLE_FORMAT == "blank_separated_columns":
        raw_df = read_table_file(
            input_path=input_file,
            sheet_name=None,
            header="infer",
        )
        long_table, group_names = parse_blank_separated_columns(
            raw_df=raw_df,
        )

    elif INPUT_TABLE_FORMAT == "long_table":
        raw_df = read_table_file(
            input_path=input_file,
            sheet_name=None,
            header="infer",
        )
        long_table, group_names = standardize_existing_long_table(
            raw_df=raw_df,
        )

    elif INPUT_TABLE_FORMAT == "graphpad_two_header":
        matrix_df = read_table_file(
            input_path=input_file,
            sheet_name="raw_for_GraphPad",
            header=None,
        )
        long_table, group_names = parse_graphpad_two_header_matrix(
            matrix_df=matrix_df,
        )

    else:
        raise ValueError(
            'INPUT_TABLE_FORMAT must be one of: "blank_separated_columns", "long_table", "graphpad_two_header".'
        )

    # Global rank transformation across all groups and all mice.
    ranked_long_table = long_table.copy()
    ranked_long_table["rank_value"] = ranked_long_table["value"].rank(method="average")

    # Confirm the helper's default random-effect column is available.
    if "mouse_id" not in long_table.columns:
        raise ValueError(
            "'mouse_id' random-effect column not found. "
            f"Available columns: {list(long_table.columns)}"
        )

    raw_random_effect_summary = build_random_effect_summary_wide_table(
        long_df=long_table,
        value_col="value",
        group_names=group_names,
        random_effect_col="mouse_id",
    )
    rank_random_effect_summary = build_random_effect_summary_wide_table(
        long_df=ranked_long_table,
        value_col="rank_value",
        group_names=group_names,
        random_effect_col="mouse_id",
    )

    # Save long/ranked tables.
    # If the input is already a long_table, do not re-save the same long table.
    # Still save ranked_long_table because ranks are recalculated globally in this notebook.
    if INPUT_TABLE_FORMAT == "long_table":
        print("INPUT_TABLE_FORMAT='long_table': skip saving parsed long_table because it is already the input table.")
    else:
        long_table.to_csv(dataset_output_dir / f"{input_file.stem}_long_table.csv", index=False, encoding="utf-8-sig")

    ranked_long_table.to_csv(dataset_output_dir / f"{input_file.stem}_ranked_long_table.csv", index=False, encoding="utf-8-sig")


    # Save GraphPad-friendly nested matrices with two header rows.
    graphpad_raw_table, graphpad_raw_summary = build_graphpad_two_header_table(
        long_df=long_table,
        value_col="value",
        group_names=group_names,
    )
    graphpad_rank_table, graphpad_rank_summary = build_graphpad_two_header_table(
        long_df=ranked_long_table,
        value_col="rank_value",
        group_names=group_names,
    )
    graphpad_paths = save_graphpad_two_header_outputs(
        raw_table=graphpad_raw_table,
        raw_summary=graphpad_raw_summary,
        rank_table=graphpad_rank_table,
        rank_summary=graphpad_rank_summary,
        output_dir=dataset_output_dir,
        prefix=input_file.stem
    )

    # Raw-value LMM.
    raw_lmm,raw_full_result,raw_null_result = run_lmm_test(
        df=long_table,
        outcome_col="value",
        group_names=group_names,
        label="raw_value_LMM",
    )

    # Rank-transformed LMM.
    rank_lmm,rank_full_result,rank_null_result = run_lmm_test(
        df=ranked_long_table,
        outcome_col="rank_value",
        group_names=group_names,
        label="rank_transformed_LMM",
    )

    # Minimal test-result workbook.
    test_results_path = dataset_output_dir / "test_results.xlsx"
    with pd.ExcelWriter(test_results_path, engine="openpyxl") as writer:
        raw_lmm["omnibus"].to_excel(writer, sheet_name="raw_lmm_omnibus", index=False)
        raw_lmm["fixed_effects"].to_excel(writer, sheet_name="raw_lmm_fixed", index=False)
        raw_lmm["pairwise"].to_excel(writer, sheet_name="raw_lmm_pairwise", index=False)
        raw_random_effect_summary.to_excel(writer, sheet_name="raw_random_effect_summary", header=False, index=False)

        rank_lmm["omnibus"].to_excel(writer, sheet_name="rank_lmm_omnibus", index=False)
        rank_lmm["fixed_effects"].to_excel(writer, sheet_name="rank_lmm_fixed", index=False)
        rank_lmm["pairwise"].to_excel(writer, sheet_name="rank_lmm_pairwise", index=False)
        rank_random_effect_summary.to_excel(writer, sheet_name="rank_random_effect_summary", header=False, index=False)

        raw_full_summary = statsmodels_summary_to_dataframe(
            raw_lmm["full_result"],
            label="raw_value_LMM_full_result",
        )
        raw_full_summary.to_excel(writer, sheet_name="raw_full_summary", index=False)

        rank_full_summary = statsmodels_summary_to_dataframe(
            rank_lmm["full_result"],
            label="rank_transformed_LMM_full_result",
        )
        rank_full_summary.to_excel(writer, sheet_name="rank_full_summary", index=False)

    dataset_summary = {
        "dataset": dataset_name,
        "input_file": str(input_file),
        "output_dir": str(dataset_output_dir),
        "n_observations": int(long_table.shape[0]),
        "n_groups": int(len(group_names)),
        "groups": ";".join(group_names),
        "random_effect_col": "mouse_id",
        "n_random_effect_units": int(long_table["mouse_id"].nunique()),
        "raw_lmm_chi2": raw_lmm["omnibus"].loc[0, "lrt_chi2"],
        "raw_lmm_df": raw_lmm["omnibus"].loc[0, "df"],
        "raw_lmm_p": raw_lmm["omnibus"].loc[0, "p"],
        "rank_lmm_chi2": rank_lmm["omnibus"].loc[0, "lrt_chi2"],
        "rank_lmm_df": rank_lmm["omnibus"].loc[0, "df"],
        "rank_lmm_p": rank_lmm["omnibus"].loc[0, "p"],
    }
    all_dataset_summaries.append(dataset_summary)

    if INPUT_TABLE_FORMAT != "long_table":
        print(f"Saved: {dataset_output_dir / f'{input_file.stem}_long_table.csv'}")

    print(f"Saved: {dataset_output_dir / f'{input_file.stem}_ranked_long_table.csv'}")
    print(f"Saved: {test_results_path}")
    print(f"Saved: {graphpad_paths['xlsx']}")

all_dataset_summaries = pd.DataFrame(all_dataset_summaries)

# In batch mode, this is still a test-result summary, not an additional analysis output.
if INPUT_MODE == "folder":
    all_dataset_summaries.to_csv(output_root / "batch_test_results_summary.csv", index=False, encoding="utf-8-sig")
    print(f"Saved batch summary: {output_root / 'batch_test_results_summary.csv'}")

all_dataset_summaries

In [ ]:
# ==============================
# Preview: mouse-level raw-value median
# ==============================

# This preview uses the last processed dataset from the main run.
# It is only for quick visual QC and is not used for statistical testing.

preview_df = long_table.copy()

mouse_summary = (
    preview_df
    .groupby(["group", "mouse_id"], observed=True)["value"]
    .median()
    .reset_index()
)

plot_data = [
    mouse_summary.loc[mouse_summary["group"] == group, "value"].dropna().values
    for group in group_names
]

plt.figure(figsize=(7, 4))
plt.boxplot(plot_data, tick_labels=group_names)
plt.ylabel("Mouse-level median of raw values")
plt.title("Preview only: mouse-level median by group")
plt.tight_layout()
plt.show()

mouse_summary.head()

## Interpretation notes

Use test_results.xlsx:

- raw_lmm_omnibus tests whether group improves the raw-value LMM compared with an intercept-only mixed model.
- rank_lmm_omnibus tests whether group improves the global-rank LMM compared with an intercept-only mixed model.
- raw_lmm_pairwise contains pairwise group contrasts on the raw-value scale.
- rank_lmm_pairwise contains pairwise group contrasts on the global-rank scale.

Input formats:

- blank_separated_columns: columns are groups and blank cells separate mice.
- long_table: helper defaults expect columns named group, mouse_id, and value; nested_subject is generated when absent.
- graphpad_two_header: helper defaults read sheet raw_for_GraphPad, with group and mouse headers in the first two rows.

GraphPad output:

- graphpad_nested_two_header.xlsx contains raw and global-rank tables plus column summaries.

Recommended reporting language for the rank-transformed model:

> To preserve rank-based analysis while accounting for non-independence among observations from the same animal, values were globally ranked across all groups and analyzed using a linear mixed-effects model with group as a fixed effect and mouse identity as a random intercept.

The runtime helper defaults to mouse_id as the random-effect unit. This is appropriate only when repeated instances of the same mouse ID refer to the same animal across groups; otherwise change the helper default to nested_subject.

In [ ]:
raw_full_result.summary()